# 03 · Embeddings locales y recuperación bilingüe

**Objetivo** — ejecutar E5 multilingual base y Qwen3 Embedding 0.6B sobre los mismos
pasajes de PI-RADS v2.1 y estudiar el efecto del idioma, del chunking y de las
instrucciones del encoder. Construiremos búsqueda exacta con NumPy y mediremos
Recall, MRR y cobertura de evidencia.

El corpus es el manual aportado por el usuario, con cobertura de uso confirmada.
Los pasajes mantienen hash, versión, páginas, secciones y offsets hacia 02.
Las anotaciones del piloto tienen revisión humana pendiente.

**En VSCode** — seleccionar `.venv/Scripts/python.exe` y ejecutar de arriba abajo.
Preparación inicial desde la raíz del proyecto:

```powershell
uv sync --locked --extra embeddings
uv run python scripts/prepare_embedding_models.py
```

En esta copia los modelos ya están preparados. La ejecución del cuaderno usa los
archivos locales. Configuramos `DEVICE="cuda:0"` y batch 1 para la RTX 2060 de 6 GB.
Las salidas incluyen el dispositivo real y el máximo de VRAM asignada.

## 1. Del tokenizer al vector

El tokenizer produce IDs y máscara de atención. El transformer calcula estados
ocultos $H\in\mathbb{R}^{L\times d}$ usando sus pesos. E5 agrega mediante media
enmascarada, $e=\sum_i m_i H_i / \sum_i m_i$. Qwen agrega el último token válido.
Después normalizamos $\hat e=e/\|e\|_2$.

E5 usa `query: ` para consultas y `passage: ` para documentos. Qwen usa una
instrucción en inglés para la consulta y el pasaje sin prefijo. La instrucción del
catálogo queda fijada antes de observar los rankings. Compararemos también consultas
sin instrucción sobre las mismas matrices de pasajes.

La revisión, dimensiones y ventana están en el catálogo. E5 admite 512 tokens con
prefijo y especiales. Qwen admite una ventana mayor, pero aquí recibe los mismos
textos preparados para E5. Ejecutamos ambos en float16 y guardamos vectores float32.
La RTX 2060 usa float16 para este experimento.

Se utiliza PyTorch y Transformers directamente para inspeccionar el forward y el
pooling. La ruta de Sentence Transformers encontró una DLL de SciPy bloqueada por
Windows durante la preparación del entorno.

Fuentes primarias del encoder y sus entradas:
[E5](https://huggingface.co/intfloat/multilingual-e5-base),
[Qwen3 Embedding](https://huggingface.co/Qwen/Qwen3-Embedding-0.6B).

In [ ]:
import gc
import json
import time
import platform
from datetime import UTC, datetime
from importlib.metadata import version
from pathlib import Path
from radiology_rag.artifacts import artifact_stage

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from IPython.display import Markdown, display

from radiology_rag.paths import project_root, within
from radiology_rag.ingestion.catalog import load_catalog, sha256_file
from radiology_rag.ingestion.chunking import TokenCounter, make_chunk
from radiology_rag.ingestion.parsing import fingerprint
from radiology_rag.retrieval.dense import (
    read_jsonl, prepare_passages, load_local_encoder, encode_cached, exact_search,
)
from radiology_rag.evaluation.retrieval import (
    resolve_evidence, evidence_in_passage, retrieval_metrics, calibrate_abstention,
)

ROOT = project_root()
DEVICE = "cuda:0"  # Cambiar explícitamente a "cpu" para otro experimento.
DTYPE = "float16"
BATCH_SIZE = 1
COLLECTION_NAMES = ["fixed_512_15", "structural_512"]
K_VALUES = (1, 3, 5)
torch.set_num_threads(2)
torch.manual_seed(0)
assert torch.cuda.is_available(), "Preparar el extra embeddings con PyTorch CUDA"
assert "sm_75" in torch.cuda.get_arch_list(), "Comprobar soporte de la RTX 2060"
display(pd.Series({
    "torch": torch.__version__, "CUDA runtime": torch.version.cuda,
    "GPU": torch.cuda.get_device_name(0), "capability": torch.cuda.get_device_capability(0),
    "VRAM MiB": torch.cuda.get_device_properties(0).total_memory // 1024**2,
    "device solicitado": DEVICE, "dtype de inferencia": DTYPE,
}))

## 2. Congelar corpus, preguntas y particiones

Leemos la ejecución completa de 02 indicada por `latest.json` y comprobamos sus
artefactos. El piloto contiene 12 casos con versión española e inglesa. Diez casos
tienen evidencia textual anotada, uno está fuera del dominio y otro pide información
visual ausente del corpus textual. Las traducciones de cada caso comparten partición.

La asignación dev/test se escribió antes de ejecutar los modelos. El conjunto de test
queda reservado para el informe final. Las anclas cortas localizadas por página
sirven para evaluar recuperación documental. Su revisión humana sigue pendiente.

In [ ]:
catalog = load_catalog(ROOT / "data/catalog/documents.v1.json")
DOCUMENT_ID = catalog["active_document_id"]
assert DOCUMENT_ID == "pirads-2.1-2019", "Este piloto pertenece a PI-RADS v2.1"
source_pointer = json.loads((ROOT / f"data/processed/{DOCUMENT_ID}/02_chunking/latest.json").read_text())
SOURCE_RUN = within(ROOT, source_pointer["path"])
source_manifest = json.loads((SOURCE_RUN / "manifest.json").read_text(encoding="utf-8"))
for name, digest in source_manifest["artifact_sha256"].items():
    assert sha256_file(within(SOURCE_RUN, name)) == digest, name
corpus_text = (SOURCE_RUN / "corpus.txt").read_text(encoding="utf-8")
blocks = read_jsonl(SOURCE_RUN / "located_blocks.jsonl")
counter = TokenCounter.from_project(ROOT)
original_collections = {name: read_jsonl(SOURCE_RUN / f"chunks_{name}.jsonl")
                        for name in COLLECTION_NAMES}
pilot_path = ROOT / f"data/evaluations/{DOCUMENT_ID}-retrieval-pilot.v1.json"
pilot = json.loads(pilot_path.read_text(encoding="utf-8"))
assert pilot["source_sha256"] == source_manifest["source_sha256"]
cases = {c["id"]: c for c in pilot["cases"]}
resolved = {key: resolve_evidence(case, blocks) for key, case in cases.items()}
queries = [{"id": f"{case['id']}-{language}", "case_id": case["id"],
            "language": language, "split": case["split"], "type": case["type"],
            "answerable": case["answerable"], "text": text}
           for case in cases.values() for language, text in case["questions"].items()]
models = json.loads((ROOT / "data/catalog/embedding_models.v1.json").read_text())["models"]
display(pd.DataFrame(queries)[["id", "language", "split", "type", "answerable"]])
print("Ejecución 02", source_manifest["run_id"], "huella del piloto", sha256_file(pilot_path))

## 3. Padres largos y pasajes que caben en el encoder

El padre estructural de la página física 43 tiene 724 tokens con E5. Lo mantenemos
en 02 y creamos hijos por ventanas de tokens, con overlap 15%, que apuntan a su
`parent_chunk_id`. Son cortes mecánicos revisables. Un corte puede separar relaciones
de una fila, incluso conservando todos sus caracteres.

Cada hijo toma un substring del corpus original. Recalculamos sus spans y comprobamos
que los intervalos cubren todo el padre. La expansión al padre completo queda disponible
para construir contexto en fases posteriores.

In [ ]:
collections = {}
preparation_rows = []
for name, original in original_collections.items():
    visible_passages = []
    for parent in original:
        if counter.count(parent["text"]) <= 512:
            visible_passages.append({**parent, "parent_chunk_id": parent["chunk_id"],
                                     "child_of_oversize_parent": False})
        else:
            intervals = []
            for a, b, _ in counter.windows(parent["text"], 512, 0.15):
                child = make_chunk(corpus_text, blocks, counter, parent["start"] + a,
                    parent["start"] + b, "embedding_child_tokens",
                    {"budget": 512, "overlap": 0.15, "parent_chunk_id": parent["chunk_id"]})
                visible_passages.append({**child, "parent_chunk_id": parent["chunk_id"],
                    "child_of_oversize_parent": True,
                    "review_status": "mechanical_table_child_human_pending"})
                intervals.append((a, b))
            cursor = 0
            for a, b in intervals:
                assert a <= cursor, "Se perdió texto del padre"
                cursor = max(cursor, b)
            assert cursor == len(parent["text"])
    reusable = prepare_passages(original, corpus_text, blocks, counter)
    assert [p["chunk_id"] for p in visible_passages] == [p["chunk_id"] for p in reusable]
    assert all(p["text"] == corpus_text[p["start"]:p["end"]] for p in visible_passages)
    assert all(counter.count(p["text"]) <= 512 for p in visible_passages)
    collections[name] = visible_passages
    preparation_rows.append({"collection": name, "original_parents": len(original),
        "passages": len(visible_passages),
        "children_of_long_parent": sum(p["child_of_oversize_parent"] for p in visible_passages)})
display(pd.DataFrame(preparation_rows))

# Las etiquetas dependen de la evidencia y los offsets, antes de calcular similitudes.
evidence_maps = {}
for name, passages in collections.items():
    evidence_maps[name] = {}
    for case_id, units in resolved.items():
        mapping = {p["chunk_id"]: evidence_in_passage(p, units) for p in passages}
        available = set().union(*mapping.values())
        assert available == {u["id"] for u in units}, (name, case_id, "Revisar el techo de cobertura")
        evidence_maps[name][case_id] = mapping
unique_passages = {p["chunk_id"]: p for ps in collections.values() for p in ps}
passage_records = [{"id": p["chunk_id"], "text": p["text"]} for p in unique_passages.values()]
print("Pasajes únicos a codificar por modelo", len(passage_records))

## 4. Forward y pooling visibles

Se comprueba el algoritmo sobre dos entradas cortas, con padding, antes de codificar
el corpus. Se comparan la implementación visible y `LocalEncoder.encode`. El resultado
se transfiere a NumPy en CPU para persistencia y búsqueda exacta.

La caché identifica modelo/revisión, hashes de archivos, textos y orden de filas,
prefijo, dispositivo, precisión, runtime CUDA y versiones de librerías. Las matrices
se publican al terminar de codificar y se verifican por SHA-256 al reutilizarlas.

In [ ]:
def visible_pooling_check(encoder, spec):
    texts = [spec["passage_prefix"] + "Prostate MRI.",
             spec["passage_prefix"] + "The index lesion is described in the reporting section."]
    batch = encoder.tokenizer(texts, padding=True, truncation=False, return_tensors="pt")
    batch = {key: value.to(encoder.device) for key, value in batch.items()}
    encoder.backbone.eval()
    with torch.inference_mode():
        H = encoder.backbone(**batch).last_hidden_state.float()
        mask = batch["attention_mask"]
        if spec["pooling"] == "mean":
            pooled = (H * mask[..., None]).sum(dim=1) / mask.sum(dim=1, keepdim=True)
        else:
            positions = torch.arange(mask.shape[1], device=mask.device)[None, :].expand_as(mask)
            last_valid = positions.masked_fill(mask == 0, -1).max(dim=1).values
            pooled = H[torch.arange(len(H), device=H.device), last_valid]
        normalized = pooled / torch.linalg.vector_norm(pooled, dim=1, keepdim=True)
        visible = normalized.cpu().numpy()
    reference = encoder.encode(texts, batch_size=2)
    error = float(np.max(np.abs(visible-reference)))
    assert np.allclose(visible, reference, atol=2e-5)
    return {"hidden_shape": list(H.shape), "embedding_shape": list(visible.shape),
            "pooling": spec["pooling"], "max_absolute_error": error}

def run_encoder_experiment(spec):
    started = time.perf_counter()
    encoder = load_local_encoder(ROOT, spec, device=DEVICE, dtype=DTYPE)
    load_seconds = time.perf_counter() - started
    try:
        check = visible_pooling_check(encoder, spec)
        matrices, reports = {}, {}
        for kind, records in [("passage", passage_records), ("query", queries), ("query_plain", queries)]:
            matrices[kind], reports[kind] = encode_cached(
                ROOT, encoder, spec, records, kind, batch_size=BATCH_SIZE)
            assert np.isfinite(matrices[kind]).all()
            assert np.allclose(np.linalg.norm(matrices[kind], axis=1), 1, atol=2e-5)
        return {"spec": spec, "matrices": matrices, "reports": reports,
                "pooling_check": check, "load_seconds": load_seconds}
    finally:
        del encoder
        gc.collect()
        torch.cuda.empty_cache()

results = {}

In [ ]:
# Modelo 1. El dispositivo real queda registrado en cada matriz.
results[models[0]["id"]] = run_encoder_experiment(models[0])
display(pd.Series(results[models[0]["id"]]["pooling_check"]))

In [ ]:
# Modelo 2, después de liberar los pesos del primero.
results[models[1]["id"]] = run_encoder_experiment(models[1])
display(pd.Series(results[models[1]["id"]]["pooling_check"]))

In [ ]:
cost_rows = []
for model_id, result in results.items():
    for kind, report in result["reports"].items():
        cost_rows.append({"model": model_id, "kind": kind, "rows": report["shape"][0],
            "dimension": report["shape"][1], "device": report["device"],
            "dtype": report["effective_dtype"], "tokens_max": max(report["token_lengths"]),
            "inference_seconds": report["inference_seconds"],
            "peak_VRAM_MiB": report["peak_vram_allocated_bytes"]/1024**2,
            "peak_process_RAM_MiB": report["peak_working_set_bytes"]/1024**2,
            "matrix_KiB": report["vectors_bytes"]/1024, "cache_hit": report["cache_hit"]})
costs = pd.DataFrame(cost_rows)
display(costs.round(3))
print("Los tiempos de una matriz reutilizada pertenecen a la creación de su caché.")
print("El máximo de RAM corresponde al proceso; la VRAM se mide por llamada de codificación.")

## 5. Búsqueda exacta con NumPy

Para vectores normalizados, $s(q,p)=\hat q\cdot\hat p=\cos(q,p)$. Con $N$ pasajes
y dimensión $d$, una consulta requiere $O(Nd)$ operaciones y almacenar $O(Nd)$ números.
Se ordenan los scores y se toman los primeros $k$. En empates conservamos el orden
de los pasajes. Las dimensiones de los dos modelos corresponden a espacios distintos.

Para este corpus pequeño, una matriz permite inspeccionar todos los scores.
Medimos la latencia de la búsqueda aparte del coste de codificar la consulta.

In [ ]:
passage_position = {record["id"]: i for i, record in enumerate(passage_records)}
rankings, metric_rows, score_rows, search_costs = [], [], [], []
for model_id, result in results.items():
    for name, passages in collections.items():
        positions = [passage_position[p["chunk_id"]] for p in passages]
        P = result["matrices"]["passage"][positions]
        for query_kind in ("query", "query_plain"):
            Q = result["matrices"][query_kind]
            started = time.perf_counter()
            scores = Q @ P.T
            order = np.argsort(-scores, axis=1, kind="stable")
            elapsed = time.perf_counter() - started
            check_indices, check_scores = exact_search(Q, P, k=5)
            assert np.array_equal(order[:, :5], check_indices)
            assert np.allclose(np.take_along_axis(scores, order[:, :5], axis=1), check_scores, atol=2e-6)
            search_costs.append({"model": model_id, "collection": name, "query_kind": query_kind,
                                 "queries": len(queries), "total_search_ms": elapsed*1000})
            for i, query in enumerate(queries):
                ranked = [passages[j]["chunk_id"] for j in order[i]]
                case_id = query["case_id"]
                mapping = evidence_maps[name][case_id]
                relevant = {identifier for identifier, units in mapping.items() if units}
                base = {"model": model_id, "collection": name, "query_kind": query_kind,
                    **{k: query[k] for k in ("id", "case_id", "language", "split", "type", "answerable")}}
                score_rows.append({**base, "top_score": float(scores[i, order[i, 0]])})
                if query["answerable"]:
                    # Fórmulas visibles para Recall, MRR y evidencia única.
                    for k in K_VALUES:
                        retrieved = ranked[:k]
                        hits = [identifier in relevant for identifier in retrieved]
                        rank = next((j+1 for j, hit in enumerate(hits) if hit), None)
                        expected = {unit["id"] for unit in resolved[case_id]}
                        found = set().union(*(mapping[identifier] for identifier in retrieved))
                        visible = {f"recall@{k}": sum(hits)/len(relevant),
                            f"precision@{k}": sum(hits)/len(retrieved),
                            f"mrr@{k}": 1/rank if rank else 0, f"hit@{k}": float(any(hits)),
                            f"evidence_recall@{k}": len(found & expected)/len(expected),
                            f"all_evidence@{k}": float(expected <= found)}
                        measured = retrieval_metrics(ranked, relevant, mapping, k=k)
                        assert measured == visible
                        metric_rows.append({**base, "k": k, **{key.split('@')[0]: value
                                            for key, value in visible.items()}})
                for rank, j in enumerate(order[i, :5], 1):
                    passage = passages[j]
                    rankings.append({**base, "rank": rank, "chunk_id": passage["chunk_id"],
                        "score": float(scores[i, j]), "pages": passage["pages"],
                        "parent_chunk_id": passage["parent_chunk_id"],
                        "evidence_units": sorted(mapping[passage["chunk_id"]])})
metrics = pd.DataFrame(metric_rows)
scores_df = pd.DataFrame(score_rows)
ranking_df = pd.DataFrame(rankings)
display(pd.DataFrame(search_costs).round(3))

## 6. Métricas, traducciones y tamaño del piloto

Recall@k cuenta pasajes relevantes anotados recuperados respecto a todos los pasajes
relevantes de esa colección. El overlap puede crear varios pasajes relevantes para
una misma unidad de evidencia. EvidenceRecall@k cuenta unidades únicas del documento
recuperadas. Reportamos ambas para estudiar el efecto de la segmentación.

MRR@k utiliza el rango del primer pasaje que contiene alguna unidad anotada.
`all_evidence@k` exige que la unión de los primeros k pasajes contenga todas las
unidades del caso. El caso multisección tiene evidencia en dos páginas.

Las dos traducciones comparten la misma anotación. El tamaño de muestra se informa
por consulta y por caso. Las consultas pareadas están correlacionadas. El piloto
permite inspeccionar fallos concretos; las anclas cortas no validan la suficiencia
de todas las condiciones de una respuesta médica.

In [ ]:
summary = (metrics.groupby(["model", "collection", "query_kind", "split", "language", "k"])
    .agg(n_queries=("id", "size"), n_cases=("case_id", "nunique"),
         recall=("recall", "mean"), mrr=("mrr", "mean"), hit=("hit", "mean"),
         evidence_recall=("evidence_recall", "mean"), all_evidence=("all_evidence", "mean"))
    .reset_index())
display(summary[(summary.query_kind == "query") & (summary.k == 5)].round(3))
paired = (metrics[(metrics.query_kind == "query") & (metrics.k == 5)]
          .pivot(index=["model", "collection", "case_id", "split"], columns="language", values="mrr"))
paired["es_minus_en"] = paired.es-paired.en
display(paired.round(3))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, column, title in zip(axes, ["evidence_recall", "mrr"],
                            ["EvidenceRecall@5 · test", "MRR@5 · test"], strict=True):
    table = summary[(summary.query_kind == "query") & (summary.k == 5) & (summary.split == "test")]
    table = table.assign(configuration=table.model+" / "+table.collection)
    table.pivot(index="configuration", columns="language", values=column).plot.bar(ax=ax)
    ax.set_title(title)
    ax.set_ylim(0, 1.05)
    ax.set_xlabel("")
    ax.tick_params(axis="x", labelrotation=30)
fig.tight_layout()
plt.show()

## 7. Controles de insuficiencia y umbral calibrado en desarrollo

Una búsqueda siempre devuelve candidatos. Para estudiar abstención, calibramos un
umbral del score máximo con los casos de desarrollo, incluyendo el control fuera
de dominio. Elegimos la mayor balanced accuracy de desarrollo, con una regla de
desempate fijada que elige el umbral mayor. Se aplica al test sin reajuste.

El control de test pregunta por un color dentro de una imagen ausente del corpus
textual. Un pasaje semánticamente parecido puede superar el umbral aunque carezca
de ese dato. Esta política requiere más controles y revisión antes de usarla en QA.

In [ ]:
abstention_rows, calibration_rows = [], []
for (model_id, name), frame in scores_df[scores_df.query_kind == "query"].groupby(["model", "collection"]):
    calibration = calibrate_abstention(frame[frame.split == "dev"].to_dict("records"))
    calibration_rows.append({"model": model_id, "collection": name, **calibration})
    for row in frame.to_dict("records"):
        predicts_answerable = row["top_score"] >= calibration["threshold"]
        abstention_rows.append({**row, "threshold": calibration["threshold"],
            "predicts_answerable": predicts_answerable,
            "scope_decision_correct": predicts_answerable == row["answerable"]})
abstention = pd.DataFrame(abstention_rows)
display(pd.DataFrame(calibration_rows).round(3))
display(abstention[(abstention.split == "test") & (~abstention.answerable)][[
    "model", "collection", "id", "top_score", "threshold", "predicts_answerable", "scope_decision_correct"]])
print("Test con preguntas respondibles y controles")
display(abstention[abstention.split == "test"].groupby(["model", "collection", "answerable"])
        .agg(n=("id", "size"), accuracy=("scope_decision_correct", "mean")))

## 8. Inspeccionar un fallo con sus páginas y offsets

Seleccionamos por una regla visible el caso de test con menor MRR@5 del E5
estructural. Revisar las anclas anotadas, sus páginas y cada pasaje devuelto.
Los fragmentos hijos conservan la referencia al padre completo. La comparación de
consultas sin instrucción permite estudiar el efecto del formato de entrada.

In [ ]:
review = metrics[(metrics.model == "e5-base") & (metrics.collection == "structural_512") &
                 (metrics.query_kind == "query") & (metrics.split == "test") & (metrics.k == 5)]
review_id = review.sort_values(["mrr", "id"]).iloc[0]["id"]
question = next(q for q in queries if q["id"] == review_id)
print(question["text"])
display(pd.DataFrame(resolved[question["case_id"]]))
rows = ranking_df[(ranking_df.model == "e5-base") & (ranking_df.collection == "structural_512") &
                  (ranking_df.query_kind == "query") & (ranking_df.id == review_id)]
display(rows[["rank", "score", "pages", "evidence_units", "chunk_id"]])
for row in rows.head(2).to_dict("records"):
    passage = unique_passages[row["chunk_id"]]
    display(pd.DataFrame(passage["spans"])[["page_number", "page_label", "region_id", "bbox_scope"]])
    print(passage["text"][:900], "\n")
display(summary[(summary.k == 5) & (summary.split == "test")][[
    "model", "collection", "query_kind", "language", "recall", "mrr", "evidence_recall"]].round(3))

## 9. Publicar matrices, filas y resultados completos

Las matrices de pasajes se guardan bajo `data/indices/`, junto a su orden de filas y
procedencia. Los informes van a `data/runs/`. Una huella vincula ejecución de 02,
piloto, modelos, dispositivo, precisión, código y entorno. Se comprueban hashes antes
de reutilizar una ejecución. Un único `latest.json` del informe referencia ambos destinos.

In [ ]:
def write_json(path, value):
    path.write_text(json.dumps(value, ensure_ascii=False, indent=2)+"\n", encoding="utf-8")

def write_jsonl(path, records):
    path.write_text("".join(json.dumps(r, ensure_ascii=False)+"\n" for r in records), encoding="utf-8")

identity = {
    "source_run_id": source_manifest["run_id"], "source_sha256": source_manifest["source_sha256"],
    "source_manifest_sha256": sha256_file(SOURCE_RUN/"manifest.json"),
    "pilot_sha256": sha256_file(pilot_path), "models": models, "device": DEVICE,
    "dtype": DTYPE, "batch_size": BATCH_SIZE, "collections": COLLECTION_NAMES,
    "environment": {p: version(p) for p in ("torch", "transformers", "numpy", "tokenizers")},
    "implementation_sha256": {p: sha256_file(ROOT/p) for p in ["uv.lock",
        "notebooks/03_embeddings_and_retrieval.ipynb", "src/radiology_rag/retrieval/dense.py",
        "src/radiology_rag/evaluation/retrieval.py"]},
    "matrix_cache_ids": {mid: {kind: r["cache_id"] for kind, r in result["reports"].items()}
                         for mid, result in results.items()},
}
RUN_ID = fingerprint(identity)[:16]
index_root = ROOT/f"data/indices/{DOCUMENT_ID}/03_dense/runs"
report_root = ROOT/f"data/runs/{DOCUMENT_ID}/03_embeddings"
index_root.mkdir(parents=True, exist_ok=True)
(report_root/"runs").mkdir(parents=True, exist_ok=True)
INDEX_OUT = index_root/RUN_ID
RUN_OUT = report_root/"runs"/RUN_ID

with (report_root/".publish.lock").open("x", encoding="utf-8") as publish_lock:
    try:
        if INDEX_OUT.exists():
            saved = json.loads((INDEX_OUT/"manifest.json").read_text(encoding="utf-8"))
            for name, digest in saved["artifact_sha256"].items():
                assert sha256_file(within(INDEX_OUT, name)) == digest
        else:
            with artifact_stage(index_root, prefix=".index-") as temporary:
                stage = Path(temporary).resolve()
                assert stage.is_relative_to((ROOT/"data/indices").resolve())
                for mid, result in results.items():
                    for name, passages in collections.items():
                        folder = stage/mid/name
                        folder.mkdir(parents=True)
                        positions = [passage_position[p["chunk_id"]] for p in passages]
                        np.save(folder/"embeddings.npy", result["matrices"]["passage"][positions], allow_pickle=False)
                        write_jsonl(folder/"passages.jsonl", passages)
                    np.save(stage/mid/"pilot_queries.npy", result["matrices"]["query"], allow_pickle=False)
                write_jsonl(stage/"queries.jsonl", queries)
                artifacts = {p.relative_to(stage).as_posix(): sha256_file(p)
                             for p in sorted(stage.rglob("*")) if p.is_file()}
                write_json(stage/"manifest.json", {"run_id": RUN_ID, "inputs": identity,
                    "artifact_sha256": artifacts, "rows_order": "passages.jsonl", "normalized": True})
                stage.rename(INDEX_OUT)
        if RUN_OUT.exists():
            report = json.loads((RUN_OUT/"manifest.json").read_text(encoding="utf-8"))
            for name, digest in report["artifact_sha256"].items():
                assert sha256_file(within(RUN_OUT, name)) == digest
        else:
            with artifact_stage(report_root/"runs", prefix=".report-") as temporary:
                stage = Path(temporary).resolve()
                assert stage.is_relative_to((ROOT/"data/runs").resolve())
                for name, frame in {"metrics": metrics, "summary": summary, "costs": costs,
                    "scores": scores_df, "abstention": abstention}.items():
                    frame.to_csv(stage/f"{name}.csv", index=False)
                write_jsonl(stage/"rankings.jsonl", rankings)
                write_json(stage/"evidence.json", resolved)
                write_json(stage/"calibration.json", calibration_rows)
                fig.savefig(stage/"retrieval_comparison.png", dpi=140)
                artifacts = {p.name: sha256_file(p) for p in sorted(stage.iterdir())}
                report = {"run_id": RUN_ID, "status": "complete", "inputs": identity,
                    "created_at": datetime.now(UTC).isoformat(),
                    "index_path": INDEX_OUT.relative_to(ROOT).as_posix(),
                    "index_manifest_sha256": sha256_file(INDEX_OUT/"manifest.json"),
                    "artifact_sha256": artifacts, "n_cases": len(cases), "n_queries": len(queries),
                    "review_status": pilot["review_status"], "encoder_executed": True,
                    "llm_calls": 0, "pooling_checks": {m: r["pooling_check"] for m,r in results.items()}}
                write_json(stage/"manifest.json", report)
                stage.rename(RUN_OUT)
        pointer = report_root/".latest.json.tmp"
        write_json(pointer, {"run_id": RUN_ID, "path": RUN_OUT.relative_to(ROOT).as_posix(),
                            "index_path": INDEX_OUT.relative_to(ROOT).as_posix()})
        pointer.replace(report_root/"latest.json")
    finally:
        publish_lock.close()
        (report_root/".publish.lock").unlink(missing_ok=True)
print("Informe completo", RUN_OUT)
print("Matrices y pasajes", INDEX_OUT)

## 10. Probar una consulta propia

Modificar `QUESTION` y ejecutar esta celda. Se usa E5 y el índice estructural publicado,
se cuentan los tokens de la consulta y se muestran pasajes con sus páginas físicas
y etiquetas impresas. Cada resultado conserva su texto y su referencia documental.

In [ ]:
QUESTION = "¿Cómo se elige la lesión índice cuando hay varias lesiones en la próstata?"
selected = next(m for m in models if m["id"] == "e5-base")
folder = INDEX_OUT/selected["id"]/"structural_512"
stored_passages = read_jsonl(folder/"passages.jsonl")
stored_vectors = np.load(folder/"embeddings.npy", allow_pickle=False)
encoder = load_local_encoder(ROOT, selected, device=DEVICE, dtype=DTYPE)
try:
    formatted = selected["query_prefix"]+QUESTION
    length = len(encoder.tokenizer(formatted, truncation=False)["input_ids"])
    assert length <= selected["encoder_max_tokens"], "La consulta excede la ventana"
    started = time.perf_counter()
    query_vector = encoder.encode([formatted])
    query_seconds = time.perf_counter()-started
    indices, values = exact_search(query_vector, stored_vectors, k=3)
    for rank, (i, value) in enumerate(zip(indices[0], values[0], strict=True), 1):
        p = stored_passages[i]
        labels = sorted({s["page_label"] for s in p["spans"]})
        print(f"\nRango {rank} · score {value:.4f} · páginas físicas {p['pages']} · impresas {labels}")
        print(p["text"][:1300])
    print("Segundos de codificación de consulta", round(query_seconds, 4))
finally:
    del encoder
    gc.collect()
    torch.cuda.empty_cache()

## 11. Interpretación y siguiente experimento

Revisar los resultados calculados por modelo, colección, idioma y partición.
Comparar MRR con EvidenceRecall, las consultas sin instrucción y los controles de
insuficiencia. La cantidad de consultas del piloto limita la generalización.
La fidelidad del parsing y las relaciones de filas siguen requiriendo revisión humana.

En 04 incorporaremos el generador y compararemos el contexto recuperado con un
baseline de documento completo cuando quepa en la ventana. Los fallos observados
deben motivar las siguientes técnicas de recuperación y construcción de contexto.

**Ejercicios** — inspeccionar las dos traducciones de un caso con rankings distintos,
revisar una etiqueta de relevancia, estudiar cuánto overlap duplica evidencia y ampliar
los controles antes de recalibrar el umbral usando exclusivamente desarrollo.